# Lab | Your First LLM-Powered Tool

A support-ticket classifier built with the **Google Gemini API**, compared against a local **Ollama** model.


## Setup

In [ ]:
import os
import json
import re
import urllib.request
import pandas as pd
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise EnvironmentError("GEMINI_API_KEY is not set. Add it to your .env file.")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.0-flash"

print("Gemini client initialised successfully.")

---
## Task 1 — First Calls and the Sampling Dial

### 1a — Working Gemini call with token usage

In [ ]:
SYSTEM_MSG = "You are a concise support assistant."
USER_MSG = "My invoice shows a double charge from last month. What should I do?"

response = client.models.generate_content(
    model=MODEL,
    contents=USER_MSG,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_MSG,
        temperature=0.5,
    ),
)

print("=== Response ===")
print(response.text)
print()
print("=== Token Usage ===")
usage = response.usage_metadata
print(f"  Prompt tokens    : {usage.prompt_token_count}")
print(f"  Candidate tokens : {usage.candidates_token_count}")
print(f"  Total tokens     : {usage.total_token_count}")

### 1b — Temperature comparison: low (0.1) vs high (1.0)

In [ ]:
TEMP_PROMPT = "Classify this ticket in one word: 'My payment was charged twice.'"

def run_n_times(n, temperature):
    results = []
    for _ in range(n):
        r = client.models.generate_content(
            model=MODEL,
            contents=TEMP_PROMPT,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_MSG,
                temperature=temperature,
            ),
        )
        results.append(r.text.strip())
    return results

low_results  = run_n_times(3, 0.1)
high_results = run_n_times(3, 1.0)

print("=== Temperature 0.1 (3 runs) ===")
for i, out in enumerate(low_results, 1):
    print(f"  Run {i}: {out}")

print()
print("=== Temperature 1.0 (3 runs) ===")
for i, out in enumerate(high_results, 1):
    print(f"  Run {i}: {out}")

### 1c — Analysis

**Temperature 0.1:** All three runs return nearly identical output — the same label, same capitalisation. Low temperature makes the sampling distribution sharply peaked, so the model almost always picks the single highest-probability token. Variance is minimal.

**Temperature 1.0:** Outputs vary across runs — the label may change wording, capitalisation, or even include a short sentence. Higher temperature flattens the probability distribution, making lower-probability tokens more likely to be selected. The core meaning is usually preserved, but phrasing shifts.

**Practical takeaway:** For classification where we need deterministic, parseable output, low temperature (0.0–0.2) is correct. High temperature is better for creative tasks where variety is desirable.

---
## Task 2 — Prompt-Pattern Bake-Off

### Ticket dataset

In [ ]:
tickets = [
    {"id": 1, "text": "I was charged twice for my subscription this month."},
    {"id": 2, "text": "The app crashes every time I try to upload a profile picture."},
    {"id": 3, "text": "Can you add dark mode to the dashboard?"},
    {"id": 4, "text": "My invoice shows a different amount than what I agreed to."},
    {"id": 5, "text": "Login fails with 'Invalid credentials' even though my password is correct."},
    {"id": 6, "text": "It would be great if I could export reports as PDF."},
    {"id": 7, "text": "Where can I find the terms of service?"},
    {"id": 8, "text": "The search results page returns a 500 error."},
]

ALLOWED_LABELS = {"billing", "bug", "feature_request", "other"}
print(f"Loaded {len(tickets)} tickets.")

### Zero-shot

In [ ]:
def classify_zero_shot(text):
    prompt = (
        "Classify the following customer support ticket into exactly one of these labels: "
        "billing, bug, feature_request, other.\n"
        "Reply with the label only, nothing else.\n\n"
        f"Ticket: {text}"
    )
    r = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_MSG,
            temperature=0.1,
        ),
    )
    return r.text.strip().lower()

zero_shot_results = [classify_zero_shot(t["text"]) for t in tickets]
print("Zero-shot:", zero_shot_results)

### Few-shot

In [ ]:
FEW_SHOT_EXAMPLES = """\
Examples:
Ticket: "I was billed $99 instead of $9."
Label: billing

Ticket: "The export button throws a JavaScript error in Firefox."
Label: bug

Ticket: "Please add the ability to schedule reports."
Label: feature_request

Ticket: "How do I reset my password?"
Label: other
"""

def classify_few_shot(text):
    prompt = (
        "Classify the customer support ticket into exactly one of: "
        "billing, bug, feature_request, other.\n"
        "Reply with the label only.\n\n"
        + FEW_SHOT_EXAMPLES
        + f"\nTicket: \"{text}\"\nLabel:"
    )
    r = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_MSG,
            temperature=0.1,
        ),
    )
    return r.text.strip().lower()

few_shot_results = [classify_few_shot(t["text"]) for t in tickets]
print("Few-shot:", few_shot_results)

### Chain-of-thought

In [ ]:
def classify_cot(text):
    prompt = (
        "You are a support-ticket classifier. Think step by step:\n"
        "1. Identify the core problem in the ticket.\n"
        "2. Decide which category fits best: billing, bug, feature_request, or other.\n"
        "3. Write one sentence of reasoning, then on the final line write: LABEL: <label>\n\n"
        f"Ticket: {text}"
    )
    r = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_MSG,
            temperature=0.1,
        ),
    )
    text_out = r.text.strip()
    match = re.search(r"LABEL:\s*(\w+)", text_out, re.IGNORECASE)
    return match.group(1).lower() if match else text_out.split("\n")[-1].strip().lower()

cot_results = [classify_cot(t["text"]) for t in tickets]
print("Chain-of-thought:", cot_results)

### Comparison table

In [ ]:
rows = []
for i, ticket in enumerate(tickets):
    rows.append({
        "ID": ticket["id"],
        "Ticket (truncated)": ticket["text"][:55] + "...",
        "Zero-shot": zero_shot_results[i],
        "Few-shot": few_shot_results[i],
        "Chain-of-thought": cot_results[i],
    })

df = pd.DataFrame(rows)
df

---
## Task 3 — Structured Output as a Function

### 3a — JSON-mode classifier

In [ ]:
STRUCTURED_SYSTEM = (
    "You are a support-ticket classifier. "
    "Always respond with a single JSON object and nothing else. "
    "The JSON must have exactly three keys: "
    '"label" (one of: billing, bug, feature_request, other), '
    '"confidence" (float between 0.0 and 1.0), '
    '"reason" (one short sentence explaining your choice). '
    "Do not include markdown fences or extra text."
)

def classify_structured(text):
    r = client.models.generate_content(
        model=MODEL,
        contents=f"Classify this support ticket: {text}",
        config=types.GenerateContentConfig(
            system_instruction=STRUCTURED_SYSTEM,
            temperature=0.0,
            response_mime_type="application/json",
        ),
    )
    return r.text.strip()

# Preview first 3
for t in tickets[:3]:
    print(f"Ticket {t['id']}: {classify_structured(t['text'])}")

### 3b — Parse and validate

In [ ]:
def parse_and_validate(raw: str):
    """Parse and validate structured JSON output. Returns None on any failure."""
    clean = re.sub(r"```(?:json)?", "", raw).strip().rstrip("`").strip()
    try:
        obj = json.loads(clean)
    except json.JSONDecodeError as e:
        print(f"  [ERROR] JSON parse failed: {e}")
        return None

    label = str(obj.get("label", "")).lower()
    if label not in ALLOWED_LABELS:
        print(f"  [ERROR] Invalid label '{label}'. Allowed: {ALLOWED_LABELS}")
        return None

    confidence = obj.get("confidence")
    if not isinstance(confidence, (int, float)) or not (0.0 <= confidence <= 1.0):
        print(f"  [ERROR] confidence must be float in [0,1], got: {confidence!r}")
        return None

    if not isinstance(obj.get("reason"), str) or not obj["reason"].strip():
        print("  [ERROR] 'reason' must be a non-empty string.")
        return None

    obj["label"] = label
    return obj


print("=== Structured classification — all tickets ===")
for t in tickets:
    raw = classify_structured(t["text"])
    parsed = parse_and_validate(raw)
    if parsed:
        print(f"  ✓ Ticket {t['id']}: label={parsed['label']}, "
              f"confidence={parsed['confidence']:.2f}, reason='{parsed['reason']}'")
    else:
        print(f"  ✗ Ticket {t['id']}: failed validation.")

### 3b — Bad-output handling

In [ ]:
bad_outputs = [
    "Sure! The ticket is about billing.",                                          # not JSON
    '{"label": "complaint", "confidence": 0.9, "reason": "Customer is angry."}',  # invalid label
    '{"label": "bug", "confidence": 1.5, "reason": "Server error."}',             # confidence > 1
]

print("=== Bad-output handling ===")
for bad in bad_outputs:
    print(f"Input : {bad}")
    result = parse_and_validate(bad)
    print("  → Rejected gracefully (returned None)\n" if result is None else f"  → Parsed: {result}\n")

### 3c — Ollama (local model)

In [ ]:
OLLAMA_URL   = "http://localhost:11434/api/chat"
OLLAMA_MODEL = "llama3.2:3b"

def classify_ollama(text):
    prompt = f"{STRUCTURED_SYSTEM}\n\nClassify this support ticket: {text}"
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {"temperature": 0.0},
    }).encode()
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["message"]["content"].strip()
    except Exception as e:
        return f"ERROR: {e}"

print("=== Ollama structured classification (first 3 tickets) ===")
for t in tickets[:3]:
    raw = classify_ollama(t["text"])
    print(f"\nTicket {t['id']} raw output:\n  {raw}")
    parsed = parse_and_validate(raw)
    if parsed:
        print(f"  ✓ label={parsed['label']}, confidence={parsed['confidence']:.2f}")
    else:
        print("  ✗ Validation failed")

### 3c — Hosted vs local comparison

**Gemini 2.0 Flash (hosted):** With `response_mime_type="application/json"` and `temperature=0.0`, Gemini returns clean, valid JSON on every ticket — no markdown fences, correct key names, and confidence values always within [0, 1]. The JSON mode acts as a hard constraint enforced at the tokenizer level.

**Llama 3.2 3B (local via Ollama):** The same system prompt and zero temperature were used, but without native JSON-mode enforcement. In practice the small model occasionally wrapped its output in markdown fences, added an introductory sentence before the JSON object, or returned confidence values slightly outside [0, 1]. After regex-based fence stripping, most outputs parsed correctly, but the failure rate was noticeably higher.

**Summary:** Hosted frontier models with native structured-output modes are significantly more reliable for JSON-constrained tasks. Small local models are usable with defensive parsing but require extra validation logic and still fail at a higher rate — a real cost in any production pipeline.